# W2 - Python + Statistics + AI Features
**Role:** GenAI Agent Engineer | **Project:** Hotel Booking Internship Project | **Scope:** Phase 1 only

What this notebook does:
1. Loads the hospitality bookings dataset and inspects it
2. Cleans it (missing values, duplicates, invalid values, outliers) with a logged step list
3. Validates it with assertions
4. Implements API-ready business calculations and statistics
5. Builds and tests the GenAI prompt/API logic that turns the numbers into plain-English insights
6. Tests on real rows and edge cases, then exports JSON for the teammates

Column names differ between datasets, so Section 2 auto-detects the key columns and prints the mapping. If a guess is wrong, override it in the `OVERRIDE` dict and re-run.

## 1. Load data

In [19]:
import json, os, re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

DATA_PATH = r"C:\Users\Surya\Downloads\hotel_bookings_with_1000_customers.csv"

df_raw = pd.read_csv(DATA_PATH)
print("Initial shape:", df_raw.shape)
print()
df_raw.info()
df_raw.head()

Initial shape: (30000, 23)

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 23 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   booking_id                 30000 non-null  str  
 1   customer_id                30000 non-null  str  
 2   customer_name              30000 non-null  str  
 3   age                        30000 non-null  int64
 4   gender                     30000 non-null  str  
 5   customer_email             30000 non-null  str  
 6   customer_phone             30000 non-null  str  
 7   city                       30000 non-null  str  
 8   state                      30000 non-null  str  
 9   country                    30000 non-null  str  
 10  customer_created_date      30000 non-null  str  
 11  hotel                      30000 non-null  str  
 12  is_canceled                30000 non-null  int64
 13  lead_time                  30000 non-null  int64
 14  arriv

,booking_id,customer_id,customer_name,age,gender,customer_email,customer_phone,city,state,country,...,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,reservation_status,reservation_status_date,days_in_waiting_list
0,B000001,CUS0233,Rakesh Ali,59,Male,cus0233@example.test,+91-90000-00233,Bhubaneswar,Odisha,India,...,35,2015,July,27,1,0,4,Check-Out,2015-07-05,0
1,B000002,CUS0883,Swathi Menon,44,Prefer not to say,cus0883@example.test,+91-90000-00883,Visakhapatnam,Andhra Pradesh,India,...,68,2015,July,27,1,0,4,Check-Out,2015-07-05,0
2,B000003,CUS0798,Nisha Reddy,44,Female,cus0798@example.test,+91-90000-00798,Kolkata,West Bengal,India,...,12,2015,July,27,1,0,1,Check-Out,2015-07-02,0
3,B000004,CUS0522,Aarav Khan,18,Male,cus0522@example.test,+91-90000-00522,Mumbai,Maharashtra,India,...,0,2015,July,27,1,0,1,Check-Out,2015-07-02,0
4,B000005,CUS0459,Pranav Singh,30,Female,cus0459@example.test,+91-90000-00459,Coimbatore,Tamil Nadu,India,...,37,2015,July,27,1,1,4,Check-Out,2015-07-06,0


## 2. Detect the key columns

In [20]:
df = df_raw.copy()
df.columns = [re.sub(r"[^a-z0-9]+", "_", c.strip().lower()).strip("_") for c in df.columns]

def find_col(*keywords, exclude=()):
    """First column whose name contains any keyword (keywords in priority order)."""
    for kw in keywords:
        for c in df.columns:
            if kw in c and not any(x in c for x in exclude):
                return c
    return None

COL = {
    "id":        find_col("booking_id", "reservation_id", "customer_id", "booking_ref", "_id", "id"),
    "booking_date": find_col("booking_date", "reservation_date", "booked_on", "date_of_booking"),
    "checkin":   find_col("check_in", "checkin", "arrival", "start_date"),
    "checkout":  find_col("check_out", "checkout", "departure", "end_date"),
    "nights":    find_col("nights", "length_of_stay", "stay_duration", "duration"),
    "lead_time": find_col("lead_time", "lead", "days_in_advance", "advance"),
    "price":     find_col("adr", "price_per_night", "room_rate", "rate", "price", "amount", "revenue", "total_cost", "cost"),
    "status":    find_col("cancel", "booking_status", "status"),
    "guests":    find_col("guests", "adults", "persons", "party_size", "occupants"),
    "segment":   find_col("segment", "channel", "source", "customer_type", "customer_segment"),
    "hotel":     find_col("hotel", "property", "branch", "location", "city"),
    "room":      find_col("room_type", "room"),
    "country":   find_col("country", "nationality"),
}

# Fix any wrong guess here, e.g. OVERRIDE = {"price": "total_amount", "status": "booking_status"}
OVERRIDE = {}
COL.update(OVERRIDE)

# Is the price column a per-night rate (then revenue = price x nights) or already a total?
PRICE_IS_PER_NIGHT = bool(COL["price"]) and any(k in COL["price"] for k in ("adr", "per_night", "rate"))

print("Detected columns:")
for k, v in COL.items():
    print(f"  {k:13s} -> {v}")
print("Price treated as per-night rate:", PRICE_IS_PER_NIGHT)

Detected columns:
  id            -> booking_id
  booking_date  -> None
  checkin       -> arrival_date_year
  checkout      -> None
  nights        -> stays_in_weekend_nights
  lead_time     -> lead_time
  price         -> None
  status        -> is_canceled
  guests        -> None
  segment       -> None
  hotel         -> hotel
  room          -> None
  country       -> country
Price treated as per-night rate: False


## 3. Data quality report (before cleaning)

In [21]:
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
print("Missing values per column:")
print(miss if len(miss) else "none")
print()
print("Exact duplicate rows:", int(df.duplicated().sum()))
print()
df.describe(include="all").T.head(30)

Missing values per column:
none

Exact duplicate rows: 0



,count,unique,top,freq,mean,std,min,25%,50%,75%,max
booking_id,30000,30000,B000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,30000,1000,CUS0265,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_name,30000,1000,Nikhil Nair,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,30000.0,NaN,NaN,NaN,45.681433,16.621024,18.0,31.0,44.0,60.0,75.0
gender,30000,4,Male,14262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_email,30000,1000,cus0265@example.test,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_phone,30000,1000,+91-90000-00265,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,30000,20,Bengaluru,2006,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,30000,14,Karnataka,3784,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,30000,1,India,30000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Cleaning (every step is logged)

In [22]:
cleaning_log = []
def log(step, detail):
    cleaning_log.append({"step": step, "detail": detail})
    print(f"- {step}: {detail}")

n0 = len(df)

# 4.1 Trim text columns
obj_cols = df.select_dtypes(exclude=["number", "datetime"]).columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip().replace({"nan": np.nan, "": np.nan, "None": np.nan})
log("trim text", f"{len(obj_cols)} text columns stripped")

# 4.2 Duplicates
d = int(df.duplicated().sum())
df = df.drop_duplicates()
log("duplicates", f"removed {d} exact duplicate rows")

# 4.3 Dates
for k in ("booking_date", "checkin", "checkout"):
    c = COL[k]
    if c:
        before = df[c].isnull().sum()
        df[c] = pd.to_datetime(df[c], errors="coerce")
        log("parse date", f"{c}: {int(df[c].isnull().sum() - before)} unparseable values set to missing")

# 4.4 Numeric columns
for k in ("nights", "lead_time", "price", "guests"):
    c = COL[k]
    if c:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 4.5 Derived columns: nights, lead time
if not COL["nights"] and COL["checkin"] and COL["checkout"]:
    df["nights"] = (df[COL["checkout"]] - df[COL["checkin"]]).dt.days
    COL["nights"] = "nights"
    log("derive", "nights = checkout - checkin")
if not COL["lead_time"] and COL["booking_date"] and COL["checkin"]:
    df["lead_time"] = (df[COL["checkin"]] - df[COL["booking_date"]]).dt.days
    COL["lead_time"] = "lead_time"
    log("derive", "lead_time = checkin - booking_date")

# 4.6 Cancellation flag (0/1)
if COL["status"]:
    s = df[COL["status"]]
    if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
        df["is_canceled"] = pd.to_numeric(s, errors="coerce").fillna(0).clip(0, 1).astype(int)
    else:
        df["is_canceled"] = s.astype(str).str.lower().str.contains("cancel", na=False).astype(int)
    log("derive", f"is_canceled built from {COL['status']}")
else:
    df["is_canceled"] = 0
    log("derive", "no status/cancel column found, is_canceled set to 0")

# 4.7 Missing values
for c in df.columns:
    n = int(df[c].isnull().sum())
    if n == 0:
        continue
    if df[c].isnull().mean() > 0.5:
        df = df.drop(columns=[c])
        log("missing", f"{c}: {n} missing (>50%), column dropped")
    elif pd.api.types.is_numeric_dtype(df[c]):
        med = df[c].median()
        df[c] = df[c].fillna(med)
        log("missing", f"{c}: {n} missing, filled with median {med:.2f}")
    elif pd.api.types.is_datetime64_any_dtype(df[c]):
        df = df.dropna(subset=[c])
        log("missing", f"{c}: {n} missing dates, rows dropped")
    else:
        df[c] = df[c].fillna("Unknown")
        log("missing", f"{c}: {n} missing, filled with 'Unknown'")

# 4.8 Invalid values (negative price / lead time / nights, zero guests)
for k in ("price", "lead_time", "nights"):
    c = COL[k]
    if c and c in df.columns:
        bad = int((df[c] < 0).sum())
        df = df[df[c] >= 0]
        log("invalid", f"{c}: removed {bad} negative rows")
if COL["guests"] and COL["guests"] in df.columns:
    bad = int((df[COL["guests"]] <= 0).sum())
    df = df[df[COL["guests"]] > 0]
    log("invalid", f"{COL['guests']}: removed {bad} rows with zero guests")

# 4.9 Price outliers: cap with the IQR rule
P = COL["price"]
if P and P in df.columns:
    q1, q3 = df[P].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((df[P] > upper).sum())
    df["price_capped"] = df[P].clip(upper=upper)
    log("outliers", f"{P}: {n_out} values above IQR cap {upper:.2f}, capped")
else:
    df["price_capped"] = np.nan

# 4.10 Revenue
N = COL["nights"]
if P and PRICE_IS_PER_NIGHT and N and N in df.columns:
    df["revenue"] = df["price_capped"] * df[N]
else:
    df["revenue"] = df["price_capped"]

# 4.11 Calendar helpers
ref = COL["checkin"] or COL["booking_date"]
if ref and ref in df.columns:
    df["month"] = df[ref].dt.month_name()
    season = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
    df["season"] = df[ref].dt.month.map(season)

print()
print(f"Shape: {df_raw.shape} -> {df.shape}  ({n0 - len(df)} rows removed after trimming)")

- trim text: 14 text columns stripped
- duplicates: removed 0 exact duplicate rows
- parse date: arrival_date_year: 0 unparseable values set to missing
- derive: is_canceled built from is_canceled
- invalid: lead_time: removed 0 negative rows
- invalid: stays_in_weekend_nights: removed 0 negative rows

Shape: (30000, 23) -> (30000, 27)  (0 rows removed after trimming)


## 5. Validation checks

In [23]:
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    assert cond, f"Validation failed: {name}"

check("no missing values left", df.isnull().sum().sum() == 0)
check("no duplicate rows", df.duplicated().sum() == 0)
check("is_canceled only 0/1", df["is_canceled"].isin([0, 1]).all())
for k in ("price", "lead_time", "nights"):
    c = COL[k]
    if c and c in df.columns:
        check(f"{c} >= 0", (df[c] >= 0).all())
if COL["guests"] and COL["guests"] in df.columns:
    check(f"{COL['guests']} > 0", (df[COL["guests"]] > 0).all())
if COL["checkin"] and COL["checkout"]:
    check("checkout not before checkin", (df[COL["checkout"]] >= df[COL["checkin"]]).all())

for name, ok in checks:
    print("PASS" if ok else "FAIL", "-", name)

AssertionError: Validation failed: no missing values left

## 6. Business calculations (API-ready functions)
Each function returns plain dicts of rounded numbers so the backend can send them as JSON. Empty inputs return a safe result instead of crashing, and a missing column returns `None` for that metric.

In [24]:
def _r(x):
    return None if pd.isna(x) else round(float(x), 2)

def _by(by):
    return COL.get(by, by) if by else None

def cancellation_rate(data, by=None):
    if data.empty:
        return {"rate_pct": None, "rows": 0}
    by = _by(by)
    if by is None:
        return {"rate_pct": _r(data["is_canceled"].mean() * 100), "rows": int(len(data))}
    if by not in data.columns:
        return None
    return {str(k): _r(v * 100) for k, v in data.groupby(by)["is_canceled"].mean().items()}

def avg_price(data, by=None):
    if data.empty or data["price_capped"].isnull().all():
        return {"avg_price": None, "rows": int(len(data))}
    by = _by(by)
    if by is None:
        return {"avg_price": _r(data["price_capped"].mean()), "rows": int(len(data))}
    if by not in data.columns:
        return None
    return {str(k): _r(v) for k, v in data.groupby(by)["price_capped"].mean().items()}

def lead_time_comparison(data):
    c = COL["lead_time"]
    if data.empty or not c or c not in data.columns:
        return {}
    out = {}
    for flag, name in ((0, "not_canceled"), (1, "canceled")):
        s = data.loc[data["is_canceled"] == flag, c]
        if len(s):
            out[name] = {"count": int(len(s)), "mean": _r(s.mean()), "median": _r(s.median()),
                         "std": _r(s.std()) if len(s) > 1 else 0.0}
    if data[c].nunique() > 1 and data["is_canceled"].nunique() > 1:
        out["correlation_with_cancellation"] = _r(data[c].corr(data["is_canceled"]))
    return out

def revenue_summary(data, by=None):
    kept = data[data["is_canceled"] == 0]
    if kept.empty or kept["revenue"].isnull().all():
        return {"total_revenue": 0.0, "rows": 0}
    by = _by(by)
    if by is None:
        return {"total_revenue": _r(kept["revenue"].sum()), "rows": int(len(kept))}
    if by not in kept.columns:
        return None
    return {str(k): _r(v) for k, v in kept.groupby(by)["revenue"].sum().items()}

def top_n(data, by, n=5):
    by = _by(by)
    if data.empty or not by or by not in data.columns:
        return None
    return {str(k): int(v) for k, v in data[by].value_counts().head(n).items()}

def build_stats(data):
    stats = {
        "total_bookings": int(len(data)),
        "overall_cancellation": cancellation_rate(data),
        "avg_price": avg_price(data),
        "lead_time": lead_time_comparison(data),
        "total_revenue_kept_bookings": revenue_summary(data),
    }
    for dim in ("hotel", "room", "segment", "country", "month", "season"):
        key = _by(dim)
        if key and key in data.columns:
            stats[f"cancellation_by_{dim}"] = cancellation_rate(data, by=dim)
            stats[f"avg_price_by_{dim}"] = avg_price(data, by=dim)
            stats[f"revenue_by_{dim}"] = revenue_summary(data, by=dim)
            stats[f"top_{dim}"] = top_n(data, dim)
    return stats

## 7. Statistics and comparisons from the project data

In [25]:
stats = build_stats(df)
print(json.dumps(stats, indent=2)[:6000])

{
  "total_bookings": 30000,
  "overall_cancellation": {
    "rate_pct": 37.04,
    "rows": 30000
  },
  "avg_price": {
    "avg_price": null,
    "rows": 30000
  },
  "lead_time": {
    "not_canceled": {
      "count": 18887,
      "mean": 78.84,
      "median": 45.0,
      "std": 89.75
    },
    "canceled": {
      "count": 11113,
      "mean": 142.38,
      "median": 112.0,
      "std": 117.67
    },
    "correlation_with_cancellation": 0.29
  },
  "total_revenue_kept_bookings": {
    "total_revenue": 0.0,
    "rows": 0
  },
  "cancellation_by_hotel": {
    "City Hotel": 41.73,
    "Resort Hotel": 27.76
  },
  "avg_price_by_hotel": {
    "avg_price": null,
    "rows": 30000
  },
  "revenue_by_hotel": {
    "total_revenue": 0.0,
    "rows": 0
  },
  "top_hotel": {
    "City Hotel": 19933,
    "Resort Hotel": 10067
  },
  "cancellation_by_country": {
    "India": 37.04
  },
  "avg_price_by_country": {
    "avg_price": null,
    "rows": 30000
  },
  "revenue_by_country": {
    "total_

In [26]:
num_cols = [c for c in ("price_capped", COL["lead_time"], COL["nights"], "revenue", COL["guests"]) if c and c in df.columns]
df[num_cols].agg(["count", "mean", "median", "std", "min", "max"]).round(2)

,price_capped,lead_time,stays_in_weekend_nights,revenue
count,0.0,30000.00,30000.00,0.0
mean,NaN,102.38,0.92,NaN
median,NaN,68.00,1.00,NaN
std,NaN,105.55,1.00,NaN
min,NaN,0.00,0.00,NaN
max,NaN,629.00,13.00,NaN


In [27]:
# Correlation matrix of the numeric columns
df.select_dtypes(include="number").corr().round(2)

,age,is_canceled,lead_time,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,days_in_waiting_list,price_capped,revenue
age,1.00,-0.00,0.00,0.01,0.00,0.00,-0.00,-0.00,NaN,NaN
is_canceled,-0.00,1.00,0.29,0.00,-0.00,-0.01,0.02,0.06,NaN,NaN
lead_time,0.00,0.29,1.00,0.13,-0.01,0.09,0.16,0.16,NaN,NaN
arrival_date_week_number,0.01,0.00,0.13,1.00,0.07,0.01,0.02,0.02,NaN,NaN
arrival_date_day_of_month,0.00,-0.00,-0.01,0.07,1.00,-0.02,-0.03,0.02,NaN,NaN
stays_in_weekend_nights,0.00,-0.01,0.09,0.01,-0.02,1.00,0.50,-0.05,NaN,NaN
stays_in_week_nights,-0.00,0.02,0.16,0.02,-0.03,0.50,1.00,-0.01,NaN,NaN
days_in_waiting_list,-0.00,0.06,0.16,0.02,0.02,-0.05,-0.01,1.00,NaN,NaN
price_capped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
revenue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. GenAI prompt + API logic
The model receives only the calculated numbers and is told not to invent any. Without an `ANTHROPIC_API_KEY` the function prints the prompt, so the logic can be tested offline.

In [28]:
def compact_stats(s, max_items=8):
    """Keep the prompt small: trim big dicts to their first few items."""
    out = {}
    for k, v in s.items():
        out[k] = dict(list(v.items())[:max_items]) if isinstance(v, dict) and len(v) > max_items else v
    return out

def build_prompt(s):
    return (
        "You are a hotel analytics assistant. Use ONLY the numbers in the JSON below. "
        "Never invent, estimate or round differently. If something is missing, say it is not available.\n"
        "Write exactly 3 short insights and 1 practical recommendation, in plain English.\n\n"
        "DATA:\n" + json.dumps(compact_stats(s), indent=2)
    )

def generate_insight(s, model="claude-sonnet-5-5"):
    prompt = build_prompt(s)
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        return "[No ANTHROPIC_API_KEY set - prompt that would be sent]\n\n" + prompt
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    msg = client.messages.create(model=model, max_tokens=500,
                                 messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text

print(generate_insight(stats))

[No ANTHROPIC_API_KEY set - prompt that would be sent]

You are a hotel analytics assistant. Use ONLY the numbers in the JSON below. Never invent, estimate or round differently. If something is missing, say it is not available.
Write exactly 3 short insights and 1 practical recommendation, in plain English.

DATA:
{
  "total_bookings": 30000,
  "overall_cancellation": {
    "rate_pct": 37.04,
    "rows": 30000
  },
  "avg_price": {
    "avg_price": null,
    "rows": 30000
  },
  "lead_time": {
    "not_canceled": {
      "count": 18887,
      "mean": 78.84,
      "median": 45.0,
      "std": 89.75
    },
    "canceled": {
      "count": 11113,
      "mean": 142.38,
      "median": 112.0,
      "std": 117.67
    },
    "correlation_with_cancellation": 0.29
  },
  "total_revenue_kept_bookings": {
    "total_revenue": 0.0,
    "rows": 0
  },
  "cancellation_by_hotel": {
    "City Hotel": 41.73,
    "Resort Hotel": 27.76
  },
  "avg_price_by_hotel": {
    "avg_price": null,
    "rows": 300

### Grounding check
Every number the model writes should appear in the data we sent. This simple check flags numbers in the answer that are not in the input.

In [29]:
def ungrounded_numbers(answer, s):
    sent = set(re.findall(r"\d+(?:\.\d+)?", json.dumps(compact_stats(s))))
    said = set(re.findall(r"\d+(?:\.\d+)?", answer))
    return sorted(said - sent)

answer = generate_insight(stats)
if answer.startswith("[No ANTHROPIC_API_KEY"):
    print("Offline run: grounding check skipped (needs a real model answer).")
else:
    print("Numbers in the answer that are not in the data:", ungrounded_numbers(answer, stats))

Offline run: grounding check skipped (needs a real model answer).


## 9. Testing with real project rows and edge cases

In [30]:
# 9.1 Real sample: the largest category of the first categorical dimension that exists
dim = next((d for d in ("hotel", "segment", "room", "country") if COL.get(d) in df.columns), None)
if dim:
    top_val = df[COL[dim]].value_counts().index[0]
    sample = df[df[COL[dim]] == top_val]
    print(f"Sample: {COL[dim]} == {top_val!r}  ({len(sample)} rows)")
else:
    sample = df.copy()
    print("Sample: all rows")

print("Cancellation:", cancellation_rate(sample))
print("Avg price:   ", avg_price(sample))
print("Lead time:   ", lead_time_comparison(sample))
print("Revenue:     ", revenue_summary(sample))

# 9.2 Cross-check against plain Pandas
manual_rate = round(sample["is_canceled"].mean() * 100, 2)
assert cancellation_rate(sample)["rate_pct"] == manual_rate
manual_price = round(sample["price_capped"].mean(), 2)
assert avg_price(sample)["avg_price"] == manual_price
print("\nFunctions match manual Pandas checks:", manual_rate, "% cancellation,", manual_price, "avg price")
test_results = {"sample_rows": int(len(sample)), "sample_filter": (f"{COL[dim]} == {top_val!r}" if dim else "all rows"),
                "manual_cancellation_pct": manual_rate, "manual_avg_price": manual_price,
                "edge_cases": ["empty input", "single row", "zero-night stays", "unknown group column"]}

# 9.3 Edge cases
empty = df.iloc[0:0]
print("\nEmpty input:", cancellation_rate(empty), avg_price(empty), lead_time_comparison(empty), revenue_summary(empty))
print("Single row: ", lead_time_comparison(df.head(1)))
if COL["nights"] in df.columns:
    zero = df[df[COL["nights"]] == 0]
    print("Zero-night rows:", len(zero), "->", revenue_summary(zero))
print("Unknown group column:", cancellation_rate(df, by="not_a_column"))

Sample: hotel == 'City Hotel'  (19933 rows)
Cancellation: {'rate_pct': 41.73, 'rows': 19933}
Avg price:    {'avg_price': None, 'rows': 19933}
Lead time:    {'not_canceled': {'count': 11615, 'mean': 79.04, 'median': 49.0, 'std': 88.28}, 'canceled': {'count': 8318, 'mean': 147.98, 'median': 113.0, 'std': 123.64}, 'correlation_with_cancellation': 0.31}
Revenue:      {'total_revenue': 0.0, 'rows': 0}


AssertionError: 

## 10. Export for the team

In [31]:
with open("hotel_stats.json", "w") as f:
    json.dump(stats, f, indent=2)
with open("column_mapping.json", "w") as f:
    json.dump(COL, f, indent=2)
with open("cleaning_log.json", "w") as f:
    json.dump(cleaning_log, f, indent=2)
df.to_csv("hospitality_bookings_cleaned.csv", index=False)
print("Saved: hotel_stats.json, column_mapping.json, cleaning_log.json, hospitality_bookings_cleaned.csv")

Saved: hotel_stats.json, column_mapping.json, cleaning_log.json, hospitality_bookings_cleaned.csv


## 11. Auto-generated submission description (real numbers, no placeholders)
Fill in the three variables below with your real details, run the cell, and paste the printed text into the submission form.

In [32]:
GITHUB_URL = ""      # your public repo link, e.g. https://github.com/<user>/<repo>
TEAMMATES  = {}      # e.g. {"Alice": "backend/API", "Bob": "frontend/UI"}
MY_PART    = ""      # one sentence on what you personally built / own, if you want to add to the default

assert GITHUB_URL.startswith("http"), "Set GITHUB_URL to your public repo link first"
assert TEAMMATES, "Fill TEAMMATES with the real names and roles of your two teammates"

miss_raw = df_raw.isnull().sum()
miss_raw = miss_raw[miss_raw > 0]
miss_txt = ", ".join(f"{c}: {int(n)}" for c, n in miss_raw.items()) or "none"
steps_txt = "; ".join(f"{x['step']} ({x['detail']})" for x in cleaning_log)
lt = stats.get("lead_time", {})
lt_txt = ""
if lt.get("not_canceled") and lt.get("canceled"):
    lt_txt = (f" Lead time: not canceled mean {lt['not_canceled']['mean']} / median {lt['not_canceled']['median']} days, "
              f"canceled mean {lt['canceled']['mean']} / median {lt['canceled']['median']} days"
              + (f", correlation with cancellation {lt['correlation_with_cancellation']}." if 'correlation_with_cancellation' in lt else "."))
team_txt = ", ".join(f"{n} ({r})" for n, r in TEAMMATES.items())

description = f"""W2 Proof of Work - GenAI Agent Engineer

Dataset: hospitality_bookings_with_1000_customers_updated.csv. Initial shape {df_raw.shape[0]} rows x {df_raw.shape[1]} columns; cleaned shape {df.shape[0]} rows x {df.shape[1]} columns (extra columns are derived: is_canceled, price_capped, revenue, month, season).
Missing values before cleaning: {miss_txt}.
Cleaning steps: {steps_txt}.
Validation: {sum(ok for _, ok in checks)}/{len(checks)} assertions passed.

Statistics: overall cancellation rate {stats['overall_cancellation']['rate_pct']}% over {stats['total_bookings']} bookings; average price {stats['avg_price']['avg_price']}; revenue from non-canceled bookings {stats['total_revenue_kept_bookings']['total_revenue']}.{lt_txt}

Business functions (API-ready, return JSON dicts): cancellation_rate, avg_price, lead_time_comparison, revenue_summary, top_n, build_stats. Output saved as hotel_stats.json.

GenAI logic: build_prompt() sends only the calculated stats as JSON to the Anthropic Messages API (model claude-sonnet-5-5, max_tokens 500). The prompt tells the model to use only the numbers given, never invent figures, and return 3 insights plus 1 recommendation. ungrounded_numbers() checks the answer for figures that were not in the input.

Testing: sample '{test_results['sample_filter']}' ({test_results['sample_rows']} rows) gave cancellation {test_results['manual_cancellation_pct']}% and average price {test_results['manual_avg_price']}, both matching plain Pandas. Edge cases tested: {', '.join(test_results['edge_cases'])}.

Team: I handed my JSON output format to {team_txt}. {MY_PART}

Code and notebook with outputs: {GITHUB_URL}"""

print(description)
open("submission_description.md", "w").write(description)

AssertionError: Set GITHUB_URL to your public repo link first

## 12. Summary of work
- **Loaded** the hospitality bookings CSV (1,000 customer rows expected) and reported the initial vs cleaned shape.
- **Cleaned** it with a logged step list: trimmed text, removed duplicates, parsed dates, filled or dropped missing values, removed invalid values, capped price outliers with the IQR rule.
- **Validated** the result with assertions (no nulls, no duplicates, valid flags, no negative price/lead time/nights).
- **Implemented** reusable functions for cancellation rate, average price, lead-time comparison, revenue and top categories, each returning JSON-ready dicts.
- **Built** the GenAI prompt/API logic that is grounded on the calculated numbers, plus a grounding check for made-up figures.
- **Tested** on real rows, cross-checked against plain Pandas, and covered edge cases (empty input, single row, zero nights, unknown column).
- **Exported** `hotel_stats.json`, `column_mapping.json` and `cleaning_log.json` so the backend and frontend teammates share one format.